# Week 35, Day 1: Strategy

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w035d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def by_points(team):
    return (-team["points"], team["name"])

def by_name(team):
    return team["name"]

table = [{"name": "Rangers", "points": 6}, {"name": "Enyimba", "points": 9}, {"name": "Hearts", "points": 6}]
for order in [by_points, by_name]:
    print(order.__name__, [t["name"] for t in sorted(table, key=order)])

In [ ]:
from abc import ABC, abstractmethod

class Route(ABC):
    @abstractmethod
    def minutes(self, km: float) -> float: ...

class Walk(Route):
    def minutes(self, km):
        return km / 5 * 60

class Cycle(Route):
    def minutes(self, km):
        return km / 18 * 60

class Bus(Route):
    def minutes(self, km):
        return 10 + km / 25 * 60            # 10 minutes' wait, then 25 km/h

class Journey:                              # the context
    def __init__(self, km: float, route: Route):
        self.km = km
        self.route = route

    def eta(self) -> str:
        return f"{round(self.route.minutes(self.km))} min"

trip = Journey(6, Walk())
print(trip.eta())
trip.route = Bus()                          # swap the strategy while the program runs
print(trip.eta(), Journey(6, Cycle()).eta())

In [ ]:
def walk(km):
    return km / 5 * 60

def cycle(km):
    return km / 18 * 60

def bus(km):
    return 10 + km / 25 * 60

def eta(km, route=walk):
    return f"{round(route(km))} min"

print(eta(6), eta(6, bus), eta(6, cycle), eta(6, lambda km: km * 2))

In [ ]:
class Taxi:
    def __init__(self, kmh=30, pickup=5):
        self.kmh = kmh
        self.pickup = pickup

    def __call__(self, km):
        return self.pickup + km / self.kmh * 60

    def __repr__(self):
        return f"Taxi(kmh={self.kmh}, pickup={self.pickup})"

def eta(km, route):
    return f"{round(route(km))} min"

rush_hour = Taxi(kmh=15, pickup=12)
print(eta(6, Taxi()), eta(6, rush_hour), rush_hour)

In [ ]:
def walk(km):
    return km / 5 * 60

def cycle(km):
    return km / 18 * 60

def bus(km):
    return 10 + km / 25 * 60

ROUTES = {"walk": walk, "cycle": cycle, "bus": bus}

def get_route(name):
    try:
        return ROUTES[name.strip().lower()]
    except KeyError:
        raise ValueError(f"unknown route {name!r}; choose from {', '.join(sorted(ROUTES))}") from None

for choice in ["Bus", "cycle", "skate"]:
    try:
        print(choice, round(get_route(choice)(6)))
    except ValueError as err:
        print(err)

In [ ]:
ROUTES = {"walk": lambda km: km / 5 * 60, "cycle": lambda km: km / 18 * 60, "bus": lambda km: 10 + km / 25 * 60}

def fastest(km):
    return min(ROUTES, key=lambda name: ROUTES[name](km))

print([(km, fastest(km)) for km in [0.3, 2, 10, 40]])

> **Key idea.** A design pattern is a named, reusable solution to a recurring design problem; the common names come from the "Gang of Four" book (1994). Strategy: a family of interchangeable ways of doing one job, each with the same interface, held and called by a context that doesn't know which it has, so new ways can be added and swapped at run time. In Python, a strategy without settings is a function; one with settings is a callable object or closure; one with several methods is a class. Choose strategies by name with a dictionary. Don't use the pattern for one or two fixed cases.

### Exercise: Shipping strategies as functions

Write three shipping strategies as **functions** that take a weight in grams and return pence: `standard` (350 up to 2000 g, then 500), `express` (double `standard`) and `collect` (always 0). Then write `quote(grams, method=standard)`, returning the text `£3.50`.

In [ ]:
___

def quote(grams, method=standard):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_strategies():\n    \"strategies\"\n    assert [standard(g) for g in [1, 2000, 2001]] == [350, 350, 500] and express(500) == 700 and express(3000) == 1000 and collect(99999) == 0\n    assert quote(500) == \"\u00a33.50\" and quote(500, express) == \"\u00a37.00\" and quote(5, collect) == \"\u00a30.00\" and quote(10, lambda g: 123) == \"\u00a31.23\"")

### Exercise: Alignment strategies

Write a dictionary `ALIGN` from `"left"`, `"right"` and `"centre"` to functions taking `(text, width)` and returning the text padded with spaces to that width (`str.ljust`, `str.rjust`, `str.center`). Then write `column(words, width, align="left")`, which returns the lines joined by newlines, choosing the strategy by name (case-insensitive). An unknown name raises `ValueError` listing the choices.

In [ ]:
___

def column(words, width, align="left"):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_align():\n    \"alignment\"\n    assert column([\"ab\", \"c\"], 4) == \"ab  \\nc   \" and column([\"ab\", \"c\"], 4, \"RIGHT\") == \"  ab\\n   c\" and column([\"ab\"], 6, \"centre\") == \"  ab  \"\n    try:\n        column([\"x\"], 3, \"justify\")\n        assert False, \"an unknown alignment should raise ValueError\"\n    except ValueError as err:\n        assert \"centre\" in str(err) and \"left\" in str(err), \"list the choices in the message\"\n    ALIGN[\"dots\"] = lambda text, width: text.ljust(width, \".\")\n    assert column([\"ab\"], 4, \"dots\") == \"ab..\"")

### Exercise: Configured strategies

A quiz app scores answers in different ways. Write three scoring strategies as **callable classes**, each called with `(correct, total, seconds)` and returning a whole number of points:

- `Flat(per_question=10)`: `per_question` for each correct answer;
- `Timed(per_question=10, bonus_under=60, bonus=20)`: like `Flat`, plus `bonus` if the quiz took fewer than `bonus_under` seconds **and** at least one answer was right;
- `Negative(right=10, wrong=-5)`: `right` for each correct answer and `wrong` for each incorrect one, never below 0 in total.

Each has a `repr` showing its settings, like `Timed(per_question=10, bonus_under=60, bonus=20)`. Then write `best_strategy(strategies, correct, total, seconds)`, which returns the strategy that gives the highest score (the first of any tie).

In [ ]:
___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_scoring():\n    \"scoring\"\n    F, T, N = ns[\"Flat\"], ns[\"Timed\"], ns[\"Negative\"]\n    assert F()(7, 10, 300) == 70 and F(5)(7, 10, 1) == 35\n    assert T()(7, 10, 59) == 90 and T()(7, 10, 60) == 70 and T()(0, 10, 1) == 0 and T(1, 100, 5)(2, 3, 50) == 7\n    assert N()(7, 10, 1) == 55 and N()(1, 10, 1) == 0 and N(4, -1)(3, 5, 1) == 10\n    assert repr(T()) == \"Timed(per_question=10, bonus_under=60, bonus=20)\" and repr(N(4, -1)) == \"Negative(right=4, wrong=-1)\" and repr(F(5)) == \"Flat(per_question=5)\"\n    s = [F(), T(), N()]\n    assert best_strategy(s, 7, 10, 30) is s[1] and best_strategy(s, 7, 10, 300) is s[0] and best_strategy([N(20, 0), F(20)], 5, 5, 1).__class__.__name__ == \"Negative\"")

### Exercise: A context that switches

Write a class `Navigator(routes, default)`: `routes` is a dictionary from a mode's name to a function giving minutes for a distance in km, and `default` is the starting mode's name (an unknown name raises `ValueError`). It has:

- `use(mode)`, which switches the current mode (an unknown name raises `ValueError` and leaves the mode unchanged);
- `eta(km)`, the minutes for the current mode, rounded to a whole number;
- `fastest(km)`, the name of the quickest mode for that distance (alphabetically first on a tie), without changing the current mode;
- a property `mode`, the current mode's name, which can't be set directly (use `use`).

In [ ]:
___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_nav():\n    \"a context\"\n    ROUTES = {\"walk\": lambda km: km / 5 * 60, \"cycle\": lambda km: km / 18 * 60, \"bus\": lambda km: 10 + km / 25 * 60}\n    N = ns[\"Navigator\"]\n    n = N(ROUTES, \"walk\")\n    assert n.mode == \"walk\" and n.eta(6) == 72\n    n.use(\"bus\")\n    assert n.mode == \"bus\" and n.eta(6) == 24 and n.fastest(1) == \"cycle\" and n.fastest(40) == \"bus\" and n.mode == \"bus\"\n    try:\n        n.use(\"skate\")\n        assert False, \"an unknown mode should raise ValueError\"\n    except ValueError:\n        pass\n    assert n.mode == \"bus\"\n    try:\n        n.mode = \"walk\"\n        assert False, \"mode should be read-only\"\n    except AttributeError:\n        pass\n    try:\n        N(ROUTES, \"fly\")\n        assert False, \"an unknown default should raise ValueError\"\n    except ValueError:\n        pass\n    assert N({\"a\": lambda km: 5, \"b\": lambda km: 5}, \"b\").fastest(1) == \"a\"")

### Bug hunt: Called too soon

`quote` should use whichever strategy it's given, but this crashes with `TypeError: 'int' object is not callable`. Fix the one line that passes the strategy, so it prints `£7.00`.

In [ ]:
def standard(grams):
    return 350

def express(grams):
    return 700

def quote(grams, method):
    pence = method(grams)
    return f"£{pence // 100}.{pence % 100:02}"

print(quote(500, express(500)))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a strategy, not its result\"\n    expect_output(\"\u00a37.00\")")

### Bug hunt: Shared between customers

Each customer has their own discount strategy object, but setting a loyalty level for one customer changes it for everyone, because the strategy keeps its setting in a class attribute. Fix `Loyalty` so this prints `[900, 1000]`.

In [ ]:
class Loyalty:
    percent = 0

    def __init__(self, percent=0):
        Loyalty.percent = percent

    def __call__(self, pence):
        return pence - pence * self.percent // 100

ada = Loyalty(10)
bo = Loyalty()
print([ada(1000), bo(1000)])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"separate settings\"\n    expect_output(\"[900, 1000]\")")

### Bug hunt: The default that never changes

The default sort order for a leaderboard is meant to be the one in `SETTINGS`, read each time `ranking` is called, but changing the setting has no effect: this prints `['Bo', 'Ada']` twice. A default argument is worked out once, when the function is defined. Fix `ranking` so it prints `['Bo', 'Ada']` and then `['Ada', 'Bo']`.

In [ ]:
ORDERS = {"points": lambda p: -p[1], "name": lambda p: p[0]}
SETTINGS = {"order": "points"}

def ranking(players, order=ORDERS[SETTINGS["order"]]):
    return [name for name, _ in sorted(players, key=order)]

players = [("Ada", 5), ("Bo", 9)]
print(ranking(players))
SETTINGS["order"] = "name"
print(ranking(players))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"read each time\"\n    expect_output(\"['Bo', 'Ada']\\n['Ada', 'Bo']\")\n    assert ranking([(\"Z\", 1), (\"A\", 2)], ORDERS[\"points\"]) == [\"A\", \"Z\"]")

### Puzzle: A context with a strategy

Arrange the lines into a `Checkout` that holds a pricing strategy and uses it, then swaps it. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
def double(x):
    return x * 2

def negate(x):
    return -x

class Add:
    def __init__(self, n):
        self.n = n
    def __call__(self, x):
        return x + self.n

def apply(x, strategy=double):
    return strategy(x)

p = ___   # apply(5)
q = ___   # apply(5, negate)
r = ___   # apply(5, Add(10))
s = ___   # [apply(1, f) for f in (double, negate, Add(1))]
t = ___   # sorted([3, -4, 2], key=negate)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r) == (10, -5, 15), \"p to r\"\n    assert s == [2, -1, 2] and t == [3, 2, -4]")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: text wrapping strategies

Write a `Paragraph(text, width, wrap="greedy", align="left")` that lays out text using two kinds of strategy, both chosen by name from dictionaries (unknown names raise `ValueError`).

**Wrapping** strategies take `(words, width)` and return a list of lines (each line a list of words), stored in `WRAPS`:

- `"greedy"`: put as many words on each line as fit (words separated by single spaces, a line's length at most `width`); a word longer than `width` goes on a line of its own.
- `"one"`: one word per line.

**Alignment** strategies take `(line_text, width, is_last)` and return the padded line, stored in `ALIGNS`:

- `"left"`: unchanged (no trailing spaces);
- `"right"`: padded on the left to `width`;
- `"justify"`: spaces added **between** words until the line is exactly `width` long, extra spaces going to the leftmost gaps first; the **last** line of a paragraph, and any line with one word, is left-aligned.

`str(paragraph)` gives the lines joined by newlines. `Paragraph` must not mention any particular strategy by name except in its default arguments.

In [ ]:
___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("TEXT = \"Welcome to the Atlas course where every week builds on the last\"\ndef test_wrap():\n    \"wrapping\"\n    words = \"the quick brown fox jumps\".split()\n    assert WRAPS[\"greedy\"](words, 10) == [[\"the\", \"quick\"], [\"brown\", \"fox\"], [\"jumps\"]]\n    assert WRAPS[\"one\"](words, 10) == [[w] for w in words] and WRAPS[\"greedy\"]([\"supercalifragilistic\", \"a\"], 5) == [[\"supercalifragilistic\"], [\"a\"]]\ndef test_layout():\n    \"layout\"\n    P = ns[\"Paragraph\"]\n    assert str(P(\"the quick brown fox jumps\", 10)) == \"the quick\\nbrown fox\\njumps\"\n    assert str(P(\"the quick brown fox jumps\", 10, align=\"right\")) == \" the quick\\n brown fox\\n     jumps\"\n    assert str(P(\"a bb ccc dd e\", 9, align=\"justify\")) == \"a  bb ccc\\ndd e\"\n    got = str(P(TEXT, 20, align=\"justify\"))\n    lines = got.split(\"\\n\")\n    assert all(len(l) == 20 for l in lines[:-1]) and lines[1] == \"course  where  every\" and lines[2] == \"week  builds  on the\" and not lines[-1].endswith(\" \"), \"got:\\n\" + got\n    assert str(P(\"one two\", 3, wrap=\"one\", align=\"right\")) == \"one\\ntwo\"\n    for bad in [dict(wrap=\"balanced\"), dict(align=\"middle\")]:\n        try:\n            P(\"x\", 5, **bad)\n            assert False, f\"{bad} should raise ValueError\"\n        except ValueError:\n            pass")

## Warm-up

`sorted` does the same job each time (putting things in order), but **how** it compares them is passed in as a function. Swapping `by_points` for `by_name` changes the behaviour without touching `sorted` at all. That's the Strategy pattern, and you've used it every time you've written `key=`.

## Design patterns

A **design pattern** is a named, reusable solution to a problem that keeps coming up in software design. It isn't code you copy, but a shape for your code, described well enough that you can recognise when it fits and talk about it in a word: "let's make the pricing a strategy" says a lot in five words.

The common names come mostly from one book, *Design Patterns: Elements of Reusable Object-Oriented Software* (1994) by Erich Gamma, Richard Helm, Ralph Johnson and John Vlissides, often called the "Gang of Four". It described 23 patterns for languages like C++ and Java. Some of those patterns exist to work around limits those languages had at the time, and in Python they shrink to almost nothing, because Python has first-class functions, duck typing and dynamic classes. This week you'll see each pattern's classic form, then its Python form, which is usually smaller.

| Day | Pattern | Problem it solves | You've met it in |
|---|---|---|---|
| 1 | Strategy | swap how a job is done | `key=` functions, Week 30's pricing rules |
| 2 | Observer | let many parts react to an event | Week 33's `Signal` |
| 3 | Factory | decide which class to create | Week 30's `KINDS` dictionary, `from_text` |
| 4 | Adapter | fit something to the interface you need | Week 32's payment adapters |
| 5 | Decorator | add behaviour by wrapping | Week 30's wrappers, Week 31's `Remember` |

## The Strategy pattern

The Gang of Four describe Strategy as: "define a family of algorithms, encapsulate each one, and make them interchangeable. Strategy lets the algorithm vary independently from clients that use it." In plainer words:

- There's **one job** with **several ways** of doing it (ways to price, route, compress, sort, validate).
- Each way is a separate object with the **same interface**.
- The code that needs the job done (the **context**) holds one of them and calls it, without knowing which.

Here's the classic shape, with classes:

The benefits are the ones from Week 34: `Journey` never changes when a new way to travel is added (open-closed), each strategy has one job (single responsibility), and each can be tested on its own.

## Strategies in Python

In Python, a strategy with a **single method and no settings** is just a function. The classes above can shrink to:

That's the same pattern with less ceremony: a function is an object with a single "method" (calling it), and Python passes functions around like any other value (Week 7). When a strategy needs **settings**, use a class with `__call__` (Week 31) or a closure, so it still looks like a function to the context:

A rough guide to which form to choose:

| The strategy... | Write it as |
|---|---|
| is one calculation with no settings | a function (or a lambda, if it's tiny) |
| has settings, or needs a helpful repr | a class with `__call__`, or a closure |
| has several methods that belong together (`price` and `describe`, say) | a class, with a Protocol or ABC describing them |

## Choosing a strategy

Strategies are often chosen by **name**, from a setting, a command-line option or a user's choice. A dictionary from name to strategy does it, with a helpful error for unknown names:

The program can also choose a strategy **for** the user: the fastest route for a given distance, say, by trying each one:

## When not to

A pattern is a tool, not a goal. Don't reach for Strategy when:

- there's only one way of doing the job, and no real sign of a second (Week 34's YAGNI);
- there are two simple cases that will never grow: an `if` is clearer than a family of classes;
- the "strategies" differ only by a number: pass the number (`discount(price, percent)`), not a strategy.

Do reach for it when the ways of doing a job are multiplying, when callers need to choose, or when an `if`/`elif` chain on a "mode" is appearing in several places.

## Practice

## Quiz

## Daily challenge

## Before you go

Find two places in your own code where behaviour depends on a "mode" or "type" setting, and decide whether each would be clearer as a strategy or is fine as an `if`. Tomorrow: the Observer pattern, for letting many parts of a program react when something happens.